# 🤖 Fluxos Inteligentes com Python + Claude
## Etapa 2 — Extração de Texto dos PDFs

> **Objetivo:** Ler cada PDF gerado na Etapa 1, extrair o texto e as tabelas de forma estruturada, e salvar o resultado em JSON — pronto para ser classificado pelo Claude na Etapa 3.

---
**Bibliotecas desta etapa:** `pdfplumber`, `PyMuPDF (fitz)`, `json`, `pathlib`

| Biblioteca | Ponto forte | Quando usar |
|---|---|---|
| **pdfplumber** | Extrai tabelas com precisão cirúrgica | NFs, OSs, Planilhas |
| **PyMuPDF** | Texto limpo e metadados do arquivo | Relatórios, textos corridos |

## 📦 1. Instalação das dependências

In [7]:
!pip install pdfplumber pymupdf --quiet
print('✅ pdfplumber e PyMuPDF instalados!')

✅ pdfplumber e PyMuPDF instalados!


## 📂 2. Verificando os PDFs da Etapa 1

> ⚠️ **Pré-requisito:** Execute o notebook da **Etapa 1** antes deste. Os PDFs devem estar em `pdfs_entrada/`.

In [8]:
from pathlib import Path
import json

PASTA_ENTRADA    = Path('pdfs_entrada')
PASTA_PROCESSADO = Path('pdfs_processados')
PASTA_RESULTADOS = Path('resultados')

# Garante que as pastas existem (caso rode isolado)
PASTA_PROCESSADO.mkdir(exist_ok=True)
PASTA_RESULTADOS.mkdir(exist_ok=True)

pdfs = sorted(PASTA_ENTRADA.glob('*.pdf'))

print(f'📁 Pasta de entrada : {PASTA_ENTRADA}/')
print(f'📄 PDFs encontrados : {len(pdfs)}\n')

for pdf in pdfs:
    print(f'  ✔  {pdf.name}')

if not pdfs:
    print('  ❌ Nenhum PDF encontrado! Execute o notebook da Etapa 1 primeiro.')

📁 Pasta de entrada : pdfs_entrada/
📄 PDFs encontrados : 7

  ✔  NF_001_ComercioNordeste.pdf
  ✔  NF_002_IndustriaPernambucana.pdf
  ✔  OS_047_ConsultoriaAlpha.pdf
  ✔  OS_048_SupermercadoBomPreco.pdf
  ✔  OS_049_ClinicaSaudeTotal.pdf
  ✔  Planilha_Estoque_Maio2025.pdf
  ✔  Relatorio_Abril_2025.pdf


## 🔬 3. Entendendo as duas ferramentas

Antes de processar tudo de uma vez, vamos ver cada biblioteca em ação com **um PDF de exemplo**.

### 3a. pdfplumber — texto + tabelas

In [9]:
import pdfplumber

pdf_exemplo = 'pdfs_entrada/NF_001_ComercioNordeste.pdf'

with pdfplumber.open(pdf_exemplo) as pdf:
    pagina = pdf.pages[0]

    print('📄 TEXTO EXTRAÍDO (pdfplumber):')
    print('-' * 50)
    texto = pagina.extract_text()
    print(texto)

    print('\n📊 TABELAS EXTRAÍDAS (pdfplumber):')
    print('-' * 50)
    tabelas = pagina.extract_tables()
    for i, tabela in enumerate(tabelas):
        print(f'  Tabela {i+1}:')
        for linha in tabela:
            print(f'    {linha}')

📄 TEXTO EXTRAÍDO (pdfplumber):
--------------------------------------------------
NOTA FISCAL ELETRONICA - NF-e
Numero: NF-2025-001 | Serie: 001
EMITENTE
Tech Solutions Ltda — CNPJ: 12.345.678/0001-90
Rua das Inovacoes, 500 — Recife, PE — CEP 50000-000
DESTINATARIO
Comercio Nordeste SA
CNPJ: 98.765.432/0001-11 | Av. Boa Viagem, 1200 — Recife, PE
ITENS DA NOTA
Descricao Qtd Unit. (R$) Total (R$)
Notebook Dell Inspiron 15 3 3499.90 10499.70
Mouse Sem Fio Logitech 10 89.90 899.00
Suporte para Monitor 5 149.90 749.50
VALOR TOTAL: R$ 11643.20
Documento gerado automaticamente — dados fictícios para fins de demonstracao.

📊 TABELAS EXTRAÍDAS (pdfplumber):
--------------------------------------------------
  Tabela 1:
    ['Descricao', 'Qtd', 'Unit. (R$)', 'Total (R$)']
    ['Notebook Dell Inspiron 15', '3', '3499.90', '10499.70']
    ['Mouse Sem Fio Logitech', '10', '89.90', '899.00']
    ['Suporte para Monitor', '5', '149.90', '749.50']


### 3b. PyMuPDF — texto e metadados do arquivo

In [10]:
import fitz  # PyMuPDF

pdf_exemplo = 'pdfs_entrada/Relatorio_Abril_2025.pdf'

doc = fitz.open(pdf_exemplo)

print('📋 METADADOS (PyMuPDF):')
print('-' * 50)
meta = doc.metadata
for chave, valor in meta.items():
    if valor:
        print(f'  {chave:15s}: {valor}')

print(f'\n  Total de páginas: {doc.page_count}')

print('\n📄 TEXTO EXTRAÍDO (PyMuPDF):')
print('-' * 50)
for num_pag, pagina in enumerate(doc):
    texto = pagina.get_text()
    print(f'--- Página {num_pag + 1} ---')
    print(texto)

doc.close()

📋 METADADOS (PyMuPDF):
--------------------------------------------------
  format         : PDF 1.4
  title          : (anonymous)
  author         : (anonymous)
  subject        : (unspecified)
  creator        : (unspecified)
  producer       : ReportLab PDF Library - (opensource)
  creationDate   : D:20260529163206+00'00'
  modDate        : D:20260529163206+00'00'

  Total de páginas: 1

📄 TEXTO EXTRAÍDO (PyMuPDF):
--------------------------------------------------
--- Página 1 ---
RELATORIO OPERACIONAL
Periodo: Abril/2025 | Tech Solutions Ltda
RESUMO EXECUTIVO
Este relatorio apresenta o desempenho operacional do mes de Abril/2025. Foram emitidas 28 Notas
Fiscais, 42 Ordens de Servico foram abertas e 38 foram concluidas no periodo.
INDICADORES DO MES
Indicador
Valor
Total de Notas Fiscais emitidas
28
Faturamento bruto (R$)
187430.50
Ordens de Servico abertas
42
Ordens de Servico concluidas
38
Ticket medio O.S. (R$)
1850.00
Clientes atendidos
19
Documento gerado automaticamente — da

## ⚙️ 4. Extrator inteligente — escolhe a melhor ferramenta por tipo

A lógica é simples:
- Se o nome do arquivo indica **NF** ou **OS** → usa **pdfplumber** (captura as tabelas)
- Para **Relatório** e **Planilha** → também usa pdfplumber, com foco em texto + tabelas
- **PyMuPDF** serve de fallback caso pdfplumber retorne texto vazio

In [11]:
import pdfplumber
import fitz
import re

def detectar_tipo(nome_arquivo: str) -> str:
    """Detecta o tipo do documento pelo nome do arquivo."""
    nome = nome_arquivo.upper()
    if nome.startswith('NF_'):
        return 'nota_fiscal'
    elif nome.startswith('OS_'):
        return 'ordem_servico'
    elif 'RELATORIO' in nome:
        return 'relatorio'
    elif 'PLANILHA' in nome or 'ESTOQUE' in nome:
        return 'planilha_estoque'
    return 'desconhecido'


def extrair_com_pdfplumber(caminho: str) -> dict:
    """Extrai texto e tabelas usando pdfplumber."""
    texto_completo = []
    tabelas_completas = []

    with pdfplumber.open(caminho) as pdf:
        num_paginas = len(pdf.pages)
        for pagina in pdf.pages:
            # Texto
            texto = pagina.extract_text() or ''
            if texto.strip():
                texto_completo.append(texto.strip())

            # Tabelas
            tabelas = pagina.extract_tables()
            for tabela in tabelas:
                if tabela:
                    # Remove células None
                    tabela_limpa = [
                        [str(cel).strip() if cel else '' for cel in linha]
                        for linha in tabela
                    ]
                    tabelas_completas.append(tabela_limpa)

    return {
        'texto': '\n'.join(texto_completo),
        'tabelas': tabelas_completas,
        'num_paginas': num_paginas,
        'ferramenta': 'pdfplumber',
    }


def extrair_com_pymupdf(caminho: str) -> dict:
    """Extrai texto e metadados usando PyMuPDF (fallback)."""
    doc = fitz.open(caminho)
    texto_completo = []

    for pagina in doc:
        texto = pagina.get_text()
        if texto.strip():
            texto_completo.append(texto.strip())

    meta = {k: v for k, v in doc.metadata.items() if v}
    num_paginas = doc.page_count
    doc.close()

    return {
        'texto': '\n'.join(texto_completo),
        'tabelas': [],
        'num_paginas': num_paginas,
        'metadados': meta,
        'ferramenta': 'pymupdf',
    }


def extrair_pdf(caminho_pdf: Path) -> dict:
    """
    Função principal de extração.
    Tenta pdfplumber primeiro; usa PyMuPDF como fallback
    se o texto retornar vazio.
    """
    nome     = caminho_pdf.name
    tipo_doc = detectar_tipo(nome)

    # Tenta pdfplumber
    resultado = extrair_com_pdfplumber(str(caminho_pdf))

    # Fallback: se texto vazio, usa PyMuPDF
    if not resultado['texto'].strip():
        print(f'  ⚠️  pdfplumber retornou vazio → usando PyMuPDF')
        resultado = extrair_com_pymupdf(str(caminho_pdf))

    resultado['arquivo']  = nome
    resultado['tipo_doc'] = tipo_doc
    resultado['chars']    = len(resultado['texto'])

    return resultado


print('✅ Funções de extração definidas!')
print('   detectar_tipo()       → identifica o documento pelo nome')
print('   extrair_com_pdfplumber() → texto + tabelas')
print('   extrair_com_pymupdf()    → fallback com metadados')
print('   extrair_pdf()         → orquestra tudo automaticamente')

✅ Funções de extração definidas!
   detectar_tipo()       → identifica o documento pelo nome
   extrair_com_pdfplumber() → texto + tabelas
   extrair_com_pymupdf()    → fallback com metadados
   extrair_pdf()         → orquestra tudo automaticamente


## 🚀 5. Processando todos os PDFs

In [12]:
from datetime import datetime

todos_extraidos = []
erros = []

print(f'🔄 Iniciando extração de {len(pdfs)} PDFs...\n')
print(f'{"Arquivo":<45} {"Tipo":<20} {"Chars":>6} {"Tabelas":>7} {"Ferramenta"}')
print('-' * 100)

for pdf_path in pdfs:
    try:
        resultado = extrair_pdf(pdf_path)
        resultado['extraido_em'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        todos_extraidos.append(resultado)

        num_tabelas = len(resultado.get('tabelas', []))
        print(
            f'  ✅ {pdf_path.name:<43} '
            f'{resultado["tipo_doc"]:<20} '
            f'{resultado["chars"]:>6} '
            f'{num_tabelas:>7} '
            f'{resultado["ferramenta"]}'
        )

    except Exception as e:
        erros.append({'arquivo': pdf_path.name, 'erro': str(e)})
        print(f'  ❌ {pdf_path.name:<43} ERRO: {e}')

print('-' * 100)
print(f'\n✅ Extraídos com sucesso : {len(todos_extraidos)}')
print(f'❌ Erros                 : {len(erros)}')

🔄 Iniciando extração de 7 PDFs...

Arquivo                                       Tipo                  Chars Tabelas Ferramenta
----------------------------------------------------------------------------------------------------
  ✅ NF_001_ComercioNordeste.pdf                 nota_fiscal             539       1 pdfplumber
  ✅ NF_002_IndustriaPernambucana.pdf            nota_fiscal             557       1 pdfplumber
  ✅ OS_047_ConsultoriaAlpha.pdf                 ordem_servico           454       1 pdfplumber
  ✅ OS_048_SupermercadoBomPreco.pdf             ordem_servico           422       1 pdfplumber
  ✅ OS_049_ClinicaSaudeTotal.pdf                ordem_servico           472       1 pdfplumber
  ✅ Planilha_Estoque_Maio2025.pdf               planilha_estoque        515       1 pdfplumber
  ✅ Relatorio_Abril_2025.pdf                    relatorio               549       1 pdfplumber
----------------------------------------------------------------------------------------------------

✅ Ex

## 💾 6. Salvando os resultados em JSON

In [13]:
# ── Salva cada extração em seu próprio JSON ────────────────────────────────────
for doc in todos_extraidos:
    nome_json = doc['arquivo'].replace('.pdf', '.json')
    caminho_json = PASTA_RESULTADOS / nome_json

    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

    print(f'  💾 Salvo: {caminho_json}')

# ── Salva também um arquivo consolidado ───────────────────────────────────────
consolidado = {
    'total_documentos': len(todos_extraidos),
    'erros': erros,
    'gerado_em': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'documentos': todos_extraidos,
}

with open(PASTA_RESULTADOS / 'extracao_consolidada.json', 'w', encoding='utf-8') as f:
    json.dump(consolidado, f, ensure_ascii=False, indent=2)

print(f'\n📦 Consolidado salvo: resultados/extracao_consolidada.json')

  💾 Salvo: resultados/NF_001_ComercioNordeste.json
  💾 Salvo: resultados/NF_002_IndustriaPernambucana.json
  💾 Salvo: resultados/OS_047_ConsultoriaAlpha.json
  💾 Salvo: resultados/OS_048_SupermercadoBomPreco.json
  💾 Salvo: resultados/OS_049_ClinicaSaudeTotal.json
  💾 Salvo: resultados/Planilha_Estoque_Maio2025.json
  💾 Salvo: resultados/Relatorio_Abril_2025.json

📦 Consolidado salvo: resultados/extracao_consolidada.json


## 🔍 7. Inspecionando um resultado — ver o que o Claude vai receber

Antes de passar para a Etapa 3, é importante ver exatamente qual texto vai chegar para o Claude classificar.

In [14]:
# Escolha qualquer índice de 0 a len(todos_extraidos)-1
INDICE = 0  # ← mude aqui para inspecionar outro documento

doc = todos_extraidos[INDICE]

print('=' * 60)
print(f'  DOCUMENTO  : {doc["arquivo"]}')
print(f'  TIPO       : {doc["tipo_doc"]}')
print(f'  FERRAMENTA : {doc["ferramenta"]}')
print(f'  PÁGINAS    : {doc["num_paginas"]}')
print(f'  CARACTERES : {doc["chars"]}')
print(f'  TABELAS    : {len(doc.get("tabelas", []))}')
print('=' * 60)

print('\n📄 TEXTO EXTRAÍDO (primeiros 800 chars):')
print('-' * 60)
print(doc['texto'][:800])

if doc.get('tabelas'):
    print(f'\n📊 PRIMEIRA TABELA ({len(doc["tabelas"][0])} linhas):')
    print('-' * 60)
    for linha in doc['tabelas'][0]:
        print(f'  {linha}')

  DOCUMENTO  : NF_001_ComercioNordeste.pdf
  TIPO       : nota_fiscal
  FERRAMENTA : pdfplumber
  PÁGINAS    : 1
  CARACTERES : 539
  TABELAS    : 1

📄 TEXTO EXTRAÍDO (primeiros 800 chars):
------------------------------------------------------------
NOTA FISCAL ELETRONICA - NF-e
Numero: NF-2025-001 | Serie: 001
EMITENTE
Tech Solutions Ltda — CNPJ: 12.345.678/0001-90
Rua das Inovacoes, 500 — Recife, PE — CEP 50000-000
DESTINATARIO
Comercio Nordeste SA
CNPJ: 98.765.432/0001-11 | Av. Boa Viagem, 1200 — Recife, PE
ITENS DA NOTA
Descricao Qtd Unit. (R$) Total (R$)
Notebook Dell Inspiron 15 3 3499.90 10499.70
Mouse Sem Fio Logitech 10 89.90 899.00
Suporte para Monitor 5 149.90 749.50
VALOR TOTAL: R$ 11643.20
Documento gerado automaticamente — dados fictícios para fins de demonstracao.

📊 PRIMEIRA TABELA (4 linhas):
------------------------------------------------------------
  ['Descricao', 'Qtd', 'Unit. (R$)', 'Total (R$)']
  ['Notebook Dell Inspiron 15', '3', '3499.90', '10499.70']
  ['Mo

## 📊 8. Resumo da Etapa 2

In [15]:
total_chars   = sum(d['chars'] for d in todos_extraidos)
total_tabelas = sum(len(d.get('tabelas', [])) for d in todos_extraidos)
tipos = {}
for d in todos_extraidos:
    tipos[d['tipo_doc']] = tipos.get(d['tipo_doc'], 0) + 1

print('=' * 55)
print('  ETAPA 2 CONCLUÍDA — Extração de Texto')
print('=' * 55)
print(f'  PDFs processados : {len(todos_extraidos)}')
print(f'  Total de chars   : {total_chars:,}')
print(f'  Tabelas extraídas: {total_tabelas}')
print(f'  Erros            : {len(erros)}')
print()
print('  Tipos encontrados:')
for tipo, qtd in tipos.items():
    print(f'    • {tipo:<25} {qtd} doc(s)')
print()
print('  Arquivos gerados:')
print('    • resultados/<nome>.json        (por documento)')
print('    • resultados/extracao_consolidada.json')
print('=' * 55)
print()
print('  Próximo passo: Etapa 3 — Classificação com Claude API')
print('=' * 55)

  ETAPA 2 CONCLUÍDA — Extração de Texto
  PDFs processados : 7
  Total de chars   : 3,508
  Tabelas extraídas: 7
  Erros            : 0

  Tipos encontrados:
    • nota_fiscal               2 doc(s)
    • ordem_servico             3 doc(s)
    • planilha_estoque          1 doc(s)
    • relatorio                 1 doc(s)

  Arquivos gerados:
    • resultados/<nome>.json        (por documento)
    • resultados/extracao_consolidada.json

  Próximo passo: Etapa 3 — Classificação com Claude API
